# Download BTS On-Time Performance Data (2024)
This notebook downloads monthly flight data from BTS and counts total rows.

In [2]:
import pandas as pd
import os
import zipfile
import io
import requests
from tqdm.notebook import tqdm

# BTS download URL pattern
BASE_URL = "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_{year}_{month}.zip"

os.makedirs('data', exist_ok=True)

year = 2024
total_rows = 0
monthly_counts = []

for month in tqdm(range(1, 13), desc="Overall Progress"):
    url = BASE_URL.format(year=year, month=month)
    print(f"\n📥 Downloading {year}-{month:02d}...")
    
    try:
        # Stream download with progress bar
        resp = requests.get(url, timeout=300, stream=True)
        resp.raise_for_status()
        
        total_size = int(resp.headers.get('content-length', 0))
        chunk_size = 1024 * 1024  # 1MB chunks
        
        data = io.BytesIO()
        with tqdm(total=total_size, unit='MB', unit_scale=True, 
                  desc=f"  {year}-{month:02d}", leave=False) as pbar:
            for chunk in resp.iter_content(chunk_size=chunk_size):
                data.write(chunk)
                pbar.update(len(chunk))
        
        data.seek(0)
        with zipfile.ZipFile(data) as z:
            csv_name = z.namelist()[0]
            with z.open(csv_name) as f:
                df = pd.read_csv(f, low_memory=False)
                rows = len(df)
                total_rows += rows
                monthly_counts.append({'year': year, 'month': month, 'rows': rows})
                print(f"  ✅ {rows:,} rows | Running total: {total_rows:,}")
                
                df.to_parquet(f'data/flights_{year}_{month:02d}.parquet', index=False)
    except Exception as e:
        print(f"  ❌ FAILED: {e}")

print(f"\n{'='*50}")
print(f"🎉 Total rows for {year}: {total_rows:,}")
print(f"{'='*50}")

Overall Progress:   0%|          | 0/12 [00:00<?, ?it/s]


📥 Downloading 2024-01...


  2024-01:   0%|          | 0.00/27.6M [00:00<?, ?MB/s]

  ✅ 547,271 rows | Running total: 547,271

📥 Downloading 2024-02...


  2024-02:   0%|          | 0.00/25.1M [00:00<?, ?MB/s]

...

  ✅ 615,497 rows | Running total: 5,913,076

📥 Downloading 2024-11...


  2024-11:   0%|          | 0.00/28.1M [00:00<?, ?MB/s]

  ✅ 575,404 rows | Running total: 6,488,480

📥 Downloading 2024-12...


  2024-12:   0%|          | 0.00/30.2M [00:00<?, ?MB/s]

  ✅ 590,581 rows | Running total: 7,079,061

🎉 Total rows for 2024: 7,079,061


In [3]:
# Summary table
summary = pd.DataFrame(monthly_counts)
print(summary.to_string(index=False))
print(f"\nTotal: {summary['rows'].sum():,} rows")

 year  month   rows
 2024      1 547271
 2024      2 519221
 2024      3 591767
 2024      4 582185
 2024      5 609743
 2024      6 611132
 2024      7 634613
 2024      8 619025
 2024      9 582622
 2024     10 615497
 2024     11 575404
 2024     12 590581

Total: 7,079,061 rows


In [4]:
# Quick peek at the columns
sample = pd.read_parquet('data/flights_2024_01.parquet')
print(f"Columns ({len(sample.columns)}):")
print(sample.columns.tolist())
print(f"\nShape: {sample.shape}")
sample.head(3)

Columns (110):
['Year', 'Quarter', 'Month', 'DayofMonth', 'DayOfWeek', 'FlightDate', 'Reporting_Airline', 'DOT_ID_Reporting_Airline', 'IATA_CODE_Reporting_Airline', 'Tail_Number', 'Flight_Number_Reporting_Airline', 'OriginAirportID', 'OriginAirportSeqID', 'OriginCityMarketID', 'Origin', 'OriginCityName', 'OriginState', 'OriginStateFips', 'OriginStateName', 'OriginWac', 'DestAirportID', 'DestAirportSeqID', 'DestCityMarketID', 'Dest', 'DestCityName', 'DestState', 'DestStateFips', 'DestStateName', 'DestWac', 'CRSDepTime', 'DepTime', 'DepDelay', 'DepDelayMinutes', 'DepDel15', 'DepartureDelayGroups', 'DepTimeBlk', 'TaxiOut', 'WheelsOff', 'WheelsOn', 'TaxiIn', 'CRSArrTime', 'ArrTime', 'ArrDelay', 'ArrDelayMinutes', 'ArrDel15', 'ArrivalDelayGroups', 'ArrTimeBlk', 'Cancelled', 'CancellationCode', 'Diverted', 'CRSElapsedTime', 'ActualElapsedTime', 'AirTime', 'Flights', 'Distance', 'DistanceGroup', 'CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay', 'FirstDepTime', 

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum,Unnamed: 109
0,2024,1,1,8,1,2024-01-08,9E,20363,9E,N485PX,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024,1,1,9,2,2024-01-09,9E,20363,9E,N912XJ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2024,1,1,10,3,2024-01-10,9E,20363,9E,N918XJ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
